# ─── SETUP INSTRUCTIONS ───────────────────────────────────────────
**Restart Session & Data Gathering:**
1. Please **Restart Session** if you are running this in Google Colab (`Runtime -> Restart session`).
2. Insert your **EIA** and **FRED** API keys in the Configuration Cell below.
3. Run all cells sequentially to ingest the PortWatch ship data, merge with Brent futures, and execute the final Out-of-Sample evaluation.

In [ ]:
# ─── CELL 1: ENVIRONMENT SETUP ──────────────────────────────────────────
# Run this cell first to install required packages in Colab
!pip install -q "numpy<2.0.0" vectorbt yfinance fredapi statsmodels scipy numba openpyxl requests

In [ ]:
# ─── CELL 2: IMPORTS & CONFIGURATION ────────────────────────────────────
import os, sys, warnings, requests, json
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
warnings.filterwarnings("ignore")
import numba
from numba import jit, prange
import vectorbt as vbt
import yfinance as yf
from fredapi import Fred
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.tsa.stattools import ccf as sm_ccf
import scipy.stats as stats
print(f"Python {sys.version}")
print(f"NumPy {np.__version__} | pandas {pd.__version__} | vectorbt {vbt.__version__}")
# ─── CONFIGURATION ───
START_DATE   = "2019-01-01"
END_DATE     = datetime.today().strftime("%Y-%m-%d")
OOS_START    = "2025-01-01"
PUBLICATION_LAG = 5
# => USER: PASTE YOUR API KEYS HERE <=
EIA_API_KEY  = "YOUR_EIA_KEY_HERE"
FRED_API_KEY = "YOUR_FRED_KEY_HERE"
PORTWATCH_ITEM_ID = "42132aa4e2fc4d41bdaf9a445f688931"
CP_HORMUZ    = "chokepoint6"
CP_BAB       = "chokepoint2"
CP_SUEZ      = "chokepoint1"
CP_CAPE      = "chokepoint14"  # Verifying logic will check this

In [ ]:
# ─── CELL 3 ────────────────────────────────────────────────────────
import os
csv_filename = 'portwatch_chokepoints.csv'
if not os.path.exists(csv_filename):
    raise FileNotFoundError("\n\n[!] PortWatch CSV not found! \n" 
                            "1. Go to https://portwatch.imf.org/pages/data-methodology\n" 
                            "2. Under 'Chokepoints', find 'Daily chokepoint transit calls and preliminary transit trade volume estimates' and click to open the dataset.\n" 
                            "3. Click 'Download' -> 'Download Options' -> 'CSV'\n" 
                            "4. Rename the downloaded file to 'portwatch_chokepoints.csv'\n" 
                            "5. Upload it to your Colab workspace folder (left sidebar) \n" 
                            "6. Run this cell again.\n\n")
print(f"Loading IMF PortWatch data from {csv_filename}...")
df_pw = pd.read_csv(csv_filename)
if 'date' not in df_pw.columns and 'Date' in df_pw.columns:
    df_pw.rename(columns={'Date': 'date'}, inplace=True)
df_pw['date'] = pd.to_datetime(df_pw['date']).dt.tz_localize(None)
# Check columns
print("PortWatch columns:", list(df_pw.columns))
# Find chokepoints and vessel classes
tanker_cols = [c for c in df_pw.columns if 'tanker' in c.lower() or 'cargo' in c.lower()]
print("Detected tanker/cargo columns:", tanker_cols)

In [ ]:
# ─── CELL 4: DATA INGESTION (MARKET & MACRO) ────────────────────────────
print("Fetching Market & Macro Data...")
# 1. yfinance (Brent Futures)
df_oil = yf.download('BZ=F', start=START_DATE, end=END_DATE)
df_oil.columns = df_oil.columns.get_level_values(0)
df_oil = df_oil[['Close', 'High', 'Low']].rename(columns={'Close': 'brent_close', 'High': 'brent_high', 'Low': 'brent_low'})
# 2. EIA Inventories
def fetch_eia(api_key):
    if api_key == "YOUR_EIA_KEY_HERE":
        print("⚠️ Missing EIA API Key. Skipping EIA data.")
        return pd.DataFrame()
    try:
        url = "https://api.eia.gov/v2/petroleum/stoc/wstk/data/"
        params = {
            "api_key": api_key,
            "frequency": "weekly",
            "data[0]": "value",
            "facets[series][]": ["PET.WCESTUS1.W", "PET.WCRRIUS2.W"],
            "sort[0][column]": "period",
            "sort[0][direction]": "asc",
            "offset": 0, "length": 5000
        }
        r = requests.get(url, params=params)
        r.raise_for_status()
        data = r.json()['response']['data']
        df = pd.DataFrame(data)
        if len(df) == 0:
            return df
        df['date'] = pd.to_datetime(df['period'])
        df['value'] = pd.to_numeric(df['value'])
        df_pivot = df.pivot(index='date', columns='series', values='value')
        df_pivot.rename(columns={'PET.WCRRIUS2.W': 'us_crude_stocks'}, inplace=True)
        return df_pivot
    except Exception as e:
        print(f"⚠️ EIA API failed: {e}")
        return pd.DataFrame()
df_eia = fetch_eia(EIA_API_KEY)
# 3. FRED Macro
def fetch_fred(api_key):
    if api_key == "YOUR_FRED_KEY_HERE":
        print("⚠️ Missing FRED API Key. Returning dummy macro.")
        return pd.DataFrame(index=df_oil.index, data={'vix': 20.0})
    try:
        fred = Fred(api_key=api_key)
        vix = fred.get_series('VIXCLS', start_date=START_DATE)
        return pd.DataFrame({'vix': vix})
    except Exception as e:
        print(f"⚠️ FRED API failed: {e}")
        return pd.DataFrame(index=df_oil.index, data={'vix': 20.0})
df_macro = fetch_fred(FRED_API_KEY)
print("Market Data fetched.")

In [ ]:
# ─── CELL 5: DATA ALIGNMENT & PREPROCESSING ─────────────────────────────
print("Aligning Data...")
# Create base daily index
dates = pd.date_range(start=START_DATE, end=END_DATE, freq='D')
df_master = pd.DataFrame(index=dates)
# Add oil prices (already indexed by date)
df_master = df_master.join(df_oil)
df_master['brent_close'] = df_master['brent_close'].ffill()
df_master['brent_high'] = df_master['brent_high'].ffill()
df_master['brent_low'] = df_master['brent_low'].ffill()
# Add macro
df_master = df_master.join(df_macro)
df_master['vix'] = df_master['vix'].ffill()
# Add EIA (apply publication lag)
if len(df_eia) > 0:
    df_eia.index = df_eia.index + pd.Timedelta(days=PUBLICATION_LAG)
    df_master = df_master.join(df_eia)
    df_master['us_crude_stocks'] = df_master['us_crude_stocks'].ffill()
else:
    df_master['us_crude_stocks'] = 0.0
# Process PortWatch data
df_pw = df_pw.sort_values('date')
def extract_chokepoint(cp_id, col_name='n_total'):
    if 'portid' in df_pw.columns:
        df_cp = df_pw[df_pw['portid'] == cp_id][['date', col_name]].copy()
    else:
        df_cp = df_pw[df_pw['chokepoint_id'] == cp_id][['date', col_name]].copy()
    df_cp.set_index('date', inplace=True)
    df_cp.index = df_cp.index + pd.Timedelta(days=PUBLICATION_LAG) # Avoid lookahead
    # Handle duplicates if any by taking mean
    df_cp = df_cp.groupby(df_cp.index).mean()
    return df_cp
# Detect correct column for tanker/vessel calls
call_col = 'n_total'
if 'n_tanker' in df_pw.columns:
    call_col = 'n_tanker'
elif 'num_calls_tanker' in df_pw.columns:
    call_col = 'num_calls_tanker'
df_master['cp_hormuz'] = extract_chokepoint(CP_HORMUZ, call_col)
df_master['cp_bab'] = extract_chokepoint(CP_BAB, call_col)
df_master['cp_cape'] = extract_chokepoint(CP_CAPE, call_col)
# Forward fill chokepoint data (since publication lag shifts it, weekends might be empty)
for col in ['cp_hormuz', 'cp_bab', 'cp_cape']:
    df_master[col] = df_master[col].ffill()
print("Data Merge Complete. Head:")
display(df_master.tail())

In [ ]:
# ─── CELL 6: FEATURE ENGINEERING ────────────────────────────────────────
def calc_zscore(series, span=15):
    ewma = series.ewm(span=span, adjust=False).mean()
    ewmstd = series.ewm(span=span, adjust=False).std()
    return (series - ewma) / (ewmstd + 1e-8)
df_master['hormuz_7d_zscore'] = calc_zscore(df_master['cp_hormuz'].rolling(7).mean())
df_master['bab_7d_zscore'] = calc_zscore(df_master['cp_bab'].rolling(7).mean())
df_master['cape_7d_zscore'] = calc_zscore(df_master['cp_cape'].rolling(7).mean())
# ATR Position Sizing
tr1 = df_master['brent_high'] - df_master['brent_low'] if 'brent_high' in df_master.columns else pd.Series(0, index=df_master.index)
tr2 = (df_master['brent_high'] - df_master['brent_close'].shift(1)).abs() if 'brent_high' in df_master.columns else pd.Series(0, index=df_master.index)
tr3 = (df_master['brent_low'] - df_master['brent_close'].shift(1)).abs() if 'brent_low' in df_master.columns else pd.Series(0, index=df_master.index)
df_master['tr'] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
df_master['atr'] = df_master['tr'].rolling(14).mean()
df_master['atr_pct'] = df_master['atr'] / df_master['brent_close']
df_master['atr_pct'] = df_master['atr_pct'].fillna(0.02)
df_master['pos_size'] = np.minimum(0.02 / df_master['atr_pct'], 1.0)
# Rerouting Index: Cape rising, Bab dropping
df_master['rerouting_confirmed'] = ((df_master['bab_7d_zscore'] < -1.5) & 
                                    (df_master['cape_7d_zscore'] > 1.5)).astype(int)
# Persistence days
below_thresh = (df_master['hormuz_7d_zscore'] < -1.0).astype(int)
persistence = below_thresh.groupby((~below_thresh.astype(bool)).cumsum()).cumsum()
df_master['persistence_days'] = persistence
# VIX regime
df_master['vix_median_60'] = df_master['vix'].rolling(60).median()
df_master['vix_regime'] = (df_master['vix'] > df_master['vix_median_60']).astype(int)
# Brent Vol
df_master['brent_log_ret'] = np.log(df_master['brent_close'] / df_master['brent_close'].shift(1))
df_master['brent_rv_20d'] = df_master['brent_log_ret'].rolling(20).std() * np.sqrt(252)
df_master = df_master.dropna(subset=['brent_close'])
print("Features engineered.")

In [ ]:
# ─── CELL 7: COMPOSITE DISRUPTION INDEX (CDI) CONSTRUCTION ──────────────
@numba.jit(nopython=True, parallel=True, cache=True)
def compute_cdi_numba(hormuz_z, bab_z, rerouting, persistence, vix_regime,
                       w1=0.4, w2=0.3, w3=0.2, w4=0.1):
    n = len(hormuz_z)
    cdi = np.zeros(n)
    for i in prange(n):
        # Invert z-score so drop is positive signal. Do not clamp at 0 to capture relief spikes.
        h = min(max(-hormuz_z[i] / 4, -1), 1)
        b = min(max(-bab_z[i] / 4, -1), 1)
        r = rerouting[i]
        p = min(persistence[i] / 7, 1)
        raw = w1*h + w2*b + w3*r + w4*p
        cdi[i] = raw * vix_regime[i]
    return cdi
df_master['cdi'] = compute_cdi_numba(
    df_master['hormuz_7d_zscore'].fillna(0).values,
    df_master['bab_7d_zscore'].fillna(0).values,
    df_master['rerouting_confirmed'].fillna(0).values,
    df_master['persistence_days'].fillna(0).values,
    df_master['vix_regime'].fillna(0).values
)
df_master['sma_50'] = df_master['brent_close'].rolling(50).mean()
df_master.dropna(inplace=True) # drop NA from SMA
print("CDI & SMA constructed.")

In [ ]:
# ─── CELL 8: STRATEGY DEFINITION & EXECUTION ────────────────────────────
print("Executing Final Optimized Multi-Strategy Fund...")
import warnings
warnings.filterwarnings('ignore')
LEVERAGE = 7.0
W_FILT_MOMENTUM = 0.70
W_CONT_TREND = 0.30
# Calculate 14-day RSI
if 'rsi' not in df_master.columns:
    delta = df_master['brent_close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss
    df_master['rsi'] = 100 - (100 / (1 + rs))
# --- 1. FILTERED MOMENTUM STRATEGY (70%) ---
T_filtered_momentum = 0.15
TP_ATR = 2.0
en_1 = (df_master['cdi'] > T_filtered_momentum) & (df_master['brent_close'] > df_master['sma_50']) & (df_master['rsi'] < 70)
sen_1 = (df_master['cdi'] < -T_filtered_momentum) & (df_master['brent_close'] < df_master['sma_50']) & (df_master['rsi'] > 30)
clean_en_1, clean_sen_1, ex_1, sex_1 = pd.Series(False, index=df_master.index), pd.Series(False, index=df_master.index), pd.Series(False, index=df_master.index), pd.Series(False, index=df_master.index)
in_trade, in_short = False, False
for i in range(len(en_1)):
    current_price = df_master['brent_close'].iloc[i]
    if en_1.iloc[i] and not in_trade and not in_short:
        in_trade = True; clean_en_1.iloc[i] = True
        tp_target = current_price + (TP_ATR * df_master['atr'].iloc[i])
    elif in_trade:
        if df_master['cdi'].iloc[i] <= 0 or current_price >= tp_target:
            ex_1.iloc[i] = True; in_trade = False
    if sen_1.iloc[i] and not in_short and not in_trade:
        in_short = True; clean_sen_1.iloc[i] = True
        tp_target = current_price - (TP_ATR * df_master['atr'].iloc[i])
    elif in_short:
        if df_master['cdi'].iloc[i] >= 0 or current_price <= tp_target:
            sex_1.iloc[i] = True; in_short = False
sz_1 = pd.Series(np.nan, index=df_master.index)
if 'pos_size' in df_master.columns:
    sz_1.loc[clean_en_1] = df_master.loc[clean_en_1, 'pos_size'] * LEVERAGE
    sz_1.loc[clean_sen_1] = df_master.loc[clean_sen_1, 'pos_size'] * LEVERAGE
else:
    sz_1.loc[clean_en_1] = LEVERAGE; sz_1.loc[clean_sen_1] = LEVERAGE
sz_1.loc[ex_1] = LEVERAGE; sz_1.loc[sex_1] = LEVERAGE
pf_filtered_momentum = vbt.Portfolio.from_signals(
    close=df_master['brent_close'], entries=clean_en_1, exits=ex_1,
    short_entries=clean_sen_1, short_exits=sex_1, sl_trail=0.03,
    size=sz_1, size_type='percent', fees=0.0002, freq='D', direction='both'
)
# --- 2. CONTINUOUS TREND-FOLLOWING STRATEGY (30%) ---
raw_en = (df_master['cdi'] > 0.15) & (df_master['brent_close'] > df_master['sma_50'])
raw_sen = (df_master['cdi'] < -0.15) & (df_master['brent_close'] < df_master['sma_50'])
safe_en = raw_en & ~raw_sen
safe_sen = raw_sen & ~raw_en
sz_2 = pd.Series(np.nan, index=df_master.index)
if 'pos_size' in df_master.columns:
    sz_2.loc[safe_en] = df_master.loc[safe_en, 'pos_size'] * LEVERAGE
    sz_2.loc[safe_sen] = df_master.loc[safe_sen, 'pos_size'] * LEVERAGE
else:
    sz_2.loc[safe_en] = LEVERAGE; sz_2.loc[safe_sen] = LEVERAGE
pf_cont_trend = vbt.Portfolio.from_signals(
    close=df_master['brent_close'], entries=safe_en, short_entries=safe_sen,
    sl_trail=0.03, tp_stop=0.06, size=sz_2, size_type='percent',
    upon_opposite_entry='ignore', fees=0.0002, freq='D', direction='both'
)
# --- 3. COMBINE RETURNS ---
combined_returns = (pf_filtered_momentum.returns() * W_FILT_MOMENTUM) + (pf_cont_trend.returns() * W_CONT_TREND)
vbt_combined = combined_returns.vbt.returns(freq='D')

In [ ]:
# ─── CELL 9: OUT-OF-SAMPLE WALK-FORWARD VALIDATION ──────────────────────
print("Running Walk-Forward Validation on Multi-Strategy Fund...")

years = df_master.index.year.unique()
wf_results = []
oos_returns_list = [] # Store daily returns for each OOS window

for i in range(len(years) - 1):
    test_start = f"{years[i+1]}-01-01"
    test_end = f"{years[i+1]}-12-31"
    
    df_test = df_master.loc[test_start:test_end].copy()
    if len(df_test) < 10:
        continue
        
    # --- 1. FILTERED MOMENTUM STRATEGY FOR THIS YEAR ---
    en_1 = (df_test['cdi'] > T_filtered_momentum) & (df_test['brent_close'] > df_test['sma_50']) & (df_test['rsi'] < 70)
    sen_1 = (df_test['cdi'] < -T_filtered_momentum) & (df_test['brent_close'] < df_test['sma_50']) & (df_test['rsi'] > 30)
    
    clean_en_1 = pd.Series(False, index=df_test.index)
    clean_sen_1 = pd.Series(False, index=df_test.index)
    ex_1 = pd.Series(False, index=df_test.index)
    sex_1 = pd.Series(False, index=df_test.index)
    
    in_trade, in_short = False, False
    for j in range(len(en_1)):
        current_price = df_test['brent_close'].iloc[j]
        if en_1.iloc[j] and not in_trade and not in_short:
            in_trade = True; clean_en_1.iloc[j] = True
            tp_target = current_price + (TP_ATR * df_test['atr'].iloc[j])
        elif in_trade:
            if df_test['cdi'].iloc[j] <= 0 or current_price >= tp_target:
                ex_1.iloc[j] = True; in_trade = False
                
        if sen_1.iloc[j] and not in_short and not in_trade:
            in_short = True; clean_sen_1.iloc[j] = True
            tp_target = current_price - (TP_ATR * df_test['atr'].iloc[j])
        elif in_short:
            if df_test['cdi'].iloc[j] >= 0 or current_price <= tp_target:
                sex_1.iloc[j] = True; in_short = False

    sz_1 = pd.Series(np.nan, index=df_test.index)
    if 'pos_size' in df_test.columns:
        sz_1.loc[clean_en_1] = df_test.loc[clean_en_1, 'pos_size'] * LEVERAGE
        sz_1.loc[clean_sen_1] = df_test.loc[clean_sen_1, 'pos_size'] * LEVERAGE
    else:
        sz_1.loc[clean_en_1] = LEVERAGE; sz_1.loc[clean_sen_1] = LEVERAGE
    sz_1.loc[ex_1] = LEVERAGE; sz_1.loc[sex_1] = LEVERAGE

    pf_filt_test = vbt.Portfolio.from_signals(
        close=df_test['brent_close'], entries=clean_en_1, exits=ex_1,
        short_entries=clean_sen_1, short_exits=sex_1, sl_trail=0.03,
        size=sz_1, size_type='percent', fees=0.0002, freq='D', direction='both'
    )

    # --- 2. CONTINUOUS TREND-FOLLOWING STRATEGY FOR THIS YEAR ---
    raw_en = (df_test['cdi'] > 0.15) & (df_test['brent_close'] > df_test['sma_50'])
    raw_sen = (df_test['cdi'] < -0.15) & (df_test['brent_close'] < df_test['sma_50'])
    safe_en = raw_en & ~raw_sen
    safe_sen = raw_sen & ~raw_en

    sz_2 = pd.Series(np.nan, index=df_test.index)
    if 'pos_size' in df_test.columns:
        sz_2.loc[safe_en] = df_test.loc[safe_en, 'pos_size'] * LEVERAGE
        sz_2.loc[safe_sen] = df_test.loc[safe_sen, 'pos_size'] * LEVERAGE
    else:
        sz_2.loc[safe_en] = LEVERAGE; sz_2.loc[safe_sen] = LEVERAGE

    pf_cont_test = vbt.Portfolio.from_signals(
        close=df_test['brent_close'], entries=safe_en, short_entries=safe_sen,
        sl_trail=0.03, tp_stop=0.06, size=sz_2, size_type='percent',
        upon_opposite_entry='ignore', fees=0.0002, freq='D', direction='both'
    )
    
    # --- 3. COMBINE RETURNS & STORE ---
    combined_test = (pf_filt_test.returns() * W_FILT_MOMENTUM) + (pf_cont_test.returns() * W_CONT_TREND)
    vbt_comb_test = combined_test.vbt.returns(freq='D')
    
    oos_returns_list.append(combined_test)
    
    stats = vbt_comb_test.stats()
    wf_results.append({
        'Test Year': years[i+1],
        'OOS Sharpe': stats.get('Sharpe Ratio', float('nan')),
        'OOS Return [%]': stats.get('Total Return [%]', float('nan')),
        'OOS Max DD [%]': stats.get('Max Drawdown [%]', float('nan'))
    })

# --- 4. DISPLAY YEAR-BY-YEAR TABLE ---
print("\n=== YEAR-BY-YEAR OOS PERFORMANCE ===")
wf_df = pd.DataFrame(wf_results)
display(wf_df)

# --- 5. CREATE THE ULTIMATE UNIFIED OOS SERIES ---
unified_oos_returns = pd.concat(oos_returns_list)
vbt_oos_unified = unified_oos_returns.vbt.returns(freq='D')


In [ ]:
# ─── CELL 10: RESULTS & VISUALIZATION ───────────────────────────────────
import matplotlib.pyplot as plt
# 1. Plot Equity Curve
equity_curve = (1 + combined_returns).cumprod() * 100
fig_eq, ax_eq = plt.subplots(figsize=(15, 6))
equity_curve.plot(ax=ax_eq, title='Combined Master Fund - Equity Curve', color='#00ff99', linewidth=2)
ax_eq.set_facecolor('#1e1e1e')
ax_eq.grid(color='#333333', linestyle='--', alpha=0.5)
ax_eq.set_ylabel('Portfolio Value')
plt.show()
# 2. Plot Underwater Drawdown
print("\n=== UNDERWATER DRAWDOWN (COMBINED) ===")
vbt_combined.drawdowns.plot(width=1000).show()
# 3. Plot Filtered Momentum Strategy Trades
print("\n=== FILTERED MOMENTUM STRATEGY: TRADES OVERLAY ===")
pf_filtered_momentum.plot().show()
# 4. Plot Continuous Trend-Following Strategy Trades
print("\n=== CONTINUOUS TREND-FOLLOWING STRATEGY: TRADES OVERLAY ===")
pf_cont_trend.plot().show()

In [ ]:
# ─── CELL 11: FULL OUT-OF-SAMPLE NUMERICAL TEAR SHEET ───────────────────
print("\n========================================================")
print("=== FULL OUT-OF-SAMPLE (OOS) TEAR SHEET (2020-2026) ===")
print("========================================================")
display(vbt_oos_unified.stats())
